# Capability・Recommendation実践

このNotebookで確認すること：問題要件、4種類の判定、候補の列挙、必要引数、実際の候補生成。適合は性能の優位性を意味しません。

[Theory](../../docs/theory/13_model_selection.md) / [Guide](../../docs/guide/13_model_selection.md) / [利用方法](../../docs/capability_workflow.md)

ProblemSpec → Recommendation → Data → Model → Fit → Posterior → Acquisition → Optimization → Candidate → Evaluation

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
)
from robotorchan.models import SingleTaskGP, create_model, get_model_class, list_models
from robotorchan.acquisition import (
    create_acquisition, get_acquisition_class, list_acquisitions,
)
from robotorchan.optim import (
    create_optimizer, create_search_strategy, list_optimizers, list_search_strategies,
)
from robotorchan.optim.constraints.contracts import CandidateConstraints
from robotorchan.optim.domains.variable_space import MixedVariableSpace

torch.manual_seed(17)
dtype = torch.double
from collections import Counter
from robotorchan.workflow import (
    ProblemSpec, evaluate_components, check_compatibility, recommend,
)
from botorch.acquisition.objective import GenericMCObjective

## 1. 条件による候補集合の違い

回帰と分類、出力数と目的数、観測制約と候補制約は別々に指定します。高次元・頑健・multi-fidelity・選好などは追加要件です。判定結果の件数は、学習前に候補集合の規模を把握するために利用できます。

In [ ]:
specs = {
    "scalar": ProblemSpec(purpose="bayesian_optimization"),
    "mixed": ProblemSpec(purpose="bayesian_optimization", input_type="mixed", q=2),
    "multiobjective": ProblemSpec(
        purpose="bayesian_optimization", output_type="multi", objective_type="multi", q=2
    ),
    "outcome_constraint": ProblemSpec(
        purpose="bayesian_optimization", output_type="multi", constrained=True, q=2
    ),
    "candidate_constraint": ProblemSpec(
        purpose="bayesian_optimization", candidate_constrained=True, q=2
    ),
    "binary": ProblemSpec(purpose="active_learning", observation_kind="binary", q=2),
    "multiclass": ProblemSpec(purpose="active_learning", observation_kind="multiclass", q=2),
    "ordinal": ProblemSpec(purpose="active_learning", observation_kind="ordinal", q=2),
    "multi_fidelity": ProblemSpec(purpose="bayesian_optimization", multi_fidelity=True),
    "multi_task": ProblemSpec(purpose="bayesian_optimization", task_type="multitask"),
    "high_dimensional": ProblemSpec(purpose="bayesian_optimization", high_dimensional=True),
    "robust": ProblemSpec(purpose="bayesian_optimization", robust=True),
    "preference": ProblemSpec(purpose="bayesian_optimization", preference=True),
}
for name, spec in specs.items():
    inventory = evaluate_components(spec)
    print(name)
    for group in ("models", "acquisitions", "optimizers", "search_strategies"):
        print(" ", group, dict(Counter(item.status.value for item in getattr(inventory, group))))

## 2. 不適合理由・条件付き判定・自動列挙

q=2にanalytic EIを指定する例、分類ALに回帰モデルを指定する例を比較します。autoは候補の列挙であって自動学習ではありません。

In [ ]:
for spec, model_name, acq_name in [
    (specs["mixed"], "MixedSingleTaskGP", "LogExpectedImprovement"),
    (specs["binary"], "SingleTaskGP", "qPredictiveEntropy"),
    (specs["scalar"], "SingleTaskGP", "qLogExpectedImprovement"),
]:
    decision = check_compatibility(
        spec, model=model_name, acquisition=acq_name, optimizer="sobol"
    )
    print(decision.status, decision.reasons, decision.conditions)

alternatives = recommend(
    specs["scalar"], model="SingleTaskGP",
    acquisition="qLogExpectedImprovement", optimizer="auto",
)
for item in alternatives:
    print(item.optimizer_name, item.compatibility.status, item.required_arguments())

## 3. 単目的・候補制約・観測制約の実行

ここからは選択された構成を実際に学習します。候補制約は入力xに課し、観測制約は予測出力g(x) <= 0に課します。後者は確率的な制約であり、真の実験結果が必ず実行可能とは限りません。

In [ ]:
X = torch.linspace(0.1, 0.9, 9, dtype=dtype).unsqueeze(-1)
Y = -((X - 0.65) ** 2)
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)
grid = torch.linspace(0, 1, 101, dtype=dtype).unsqueeze(-1)
for key in ("scalar", "candidate_constraint", "outcome_constraint"):
    spec = specs[key]
    (choice,) = recommend(
        spec, model="SingleTaskGP", acquisition="qLogExpectedImprovement", optimizer="sobol"
    )
    print(key, choice.required_arguments(), choice.compatibility.conditions)
    observations = torch.cat([Y, X - 0.8], dim=-1) if key == "outcome_constraint" else Y
    model = create_model(choice.model_name, X, observations)
    model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
    with torch.no_grad():
        mean = model.posterior(grid).mean
    acq_kwargs = {"best_f": Y.max()}
    if key == "outcome_constraint":
        acq_kwargs["objective"] = GenericMCObjective(lambda samples, X=None: samples[..., 0])
        acq_kwargs["constraints"] = [lambda samples: samples[..., 1]]
    acq = create_acquisition(choice.acquisition_name, model=model, **acq_kwargs)
    optimize = create_optimizer(choice.optimizer_name, raw_samples=32, seed=17)
    options = {}
    if key == "candidate_constraint":
        options["constraints"] = CandidateConstraints(
            inequality_constraints=((torch.tensor([0]), torch.tensor([1.], dtype=dtype), 0.2),),
        )
    candidate, score = optimize(acq, bounds, q=spec.q, **options)
    assert candidate.shape == (spec.q, 1)
    if key == "candidate_constraint":
        assert torch.all(candidate >= 0.2 - 1e-8)
    print(candidate, score, -((candidate - 0.65) ** 2))
    plt.figure(figsize=(7, 3))
    plt.plot(grid.squeeze(), mean[:, 0], label="Posterior objective")
    plt.scatter(X.squeeze(), Y.squeeze(), label="Observations")
    plt.scatter(candidate.squeeze(), -((candidate.squeeze() - 0.65) ** 2), label="Candidates")
    plt.title(key)
    plt.legend()
    plt.show()

## 4. Mixed・多目的の構成選択と候補生成

各ケースの必要引数を表示し、選択された名前をFactoryへ渡します。

In [ ]:
(choice,) = recommend(
    specs["mixed"], model="MixedSingleTaskGP",
    acquisition="qLogExpectedImprovement", optimizer="sobol",
)
print(choice.required_arguments())
Xm = torch.cat([X, (torch.arange(len(X)) % 2).to(dtype).unsqueeze(-1)], dim=-1)
Ym = Y + 0.1 * Xm[:, 1:]
bm = torch.tensor([[0., 0.], [1., 1.]], dtype=dtype)
model = create_model(choice.model_name, Xm, Ym, cat_dims=[1])
model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
acq = create_acquisition(choice.acquisition_name, model=model, best_f=Ym.max())
optimize = create_optimizer(choice.optimizer_name, raw_samples=32, seed=17)
candidate, score = optimize(
    acq, bm, q=2, variable_space=MixedVariableSpace(bounds=bm, integer_dims=(1,))
)
assert candidate.shape == (2, 2)
assert torch.equal(candidate[:, 1], candidate[:, 1].round())
print(candidate, score)

In [ ]:
(choice,) = recommend(
    specs["multiobjective"], model="SingleTaskGP",
    acquisition="qLogExpectedHypervolumeImprovement", optimizer="sobol",
)
print(choice.required_arguments())
Y2 = torch.cat([Y, -((X - 0.25) ** 2)], dim=-1)
model = create_model(choice.model_name, X, Y2)
model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
ref = Y2.min(dim=0).values - 0.1
acq = create_acquisition(
    choice.acquisition_name, model=model, ref_point=ref.tolist(),
    partitioning=FastNondominatedPartitioning(ref_point=ref, Y=Y2),
    sampler=SobolQMCNormalSampler(torch.Size([16]), seed=17),
)
optimize = create_optimizer(choice.optimizer_name, raw_samples=32, seed=17)
candidate, score = optimize(acq, bounds, q=2)
assert candidate.shape == (2, 1)
new_y = torch.cat([-((candidate - 0.65) ** 2), -((candidate - 0.25) ** 2)], dim=-1)
print(candidate, score, new_y)
plt.scatter(Y2[:, 0], Y2[:, 1], label="Observations")
plt.scatter(new_y[:, 0], new_y[:, 1], label="Candidates")
plt.legend()
plt.show()

## 5. 分類AL：Search Strategyを選択する

二値・多クラス・順序分類を別の観測形式として指定します。実データでは検証データを用いて学習回数や予測確率の較正を評価してください。

In [ ]:
for key, name, k in [
    ("binary", "BinarySingleTaskGPClassifier", 2),
    ("multiclass", "MulticlassSingleTaskGPClassifier", 3),
    ("ordinal", "OrdinalSingleTaskGPClassifier", 3),
]:
    (choice,) = recommend(
        specs[key], model=name, acquisition="qPredictiveEntropy",
        search_strategy="SobolSearchStrategy",
    )
    print(choice.required_arguments())
    labels = (X.squeeze(-1) * k).long()
    kwargs = {"num_classes": k} if k > 2 else {}
    model = create_model(choice.model_name, X, labels, **kwargs)
    model.fit(epochs=2)
    with torch.no_grad():
        probabilities = model.predict_proba(grid)
    acq = create_acquisition(choice.acquisition_name, model=model)
    strategy = create_search_strategy(choice.search_strategy_name, bounds, num_samples=32, seed=17)
    candidate = strategy.optimize(acq, q=2).candidates
    assert candidate.shape == (2, 1)
    print(key, candidate)
    plt.figure(figsize=(7, 3))
    plt.plot(grid.squeeze(), probabilities)
    for x in candidate.flatten():
        plt.axvline(x.item(), linestyle="--")
    plt.title(key)
    plt.show()

## 6. 実務での適合判定と関連資料

静的な適合判定は構成選択の出発点です。学習後の予測精度、posteriorの形状、獲得関数の評価、候補点の実行可能性を実データで確認してください。

モデル固有の学習方法と候補生成については、次の実践例を参照できます。

- [Multi-Fidelity GP](03_multi_fidelity_gp.ipynb)
- [Multi-Task GP](04_multitask_gp.ipynb)
- [高次元BO](54_high_dimensional_bo_benchmark.ipynb)
- [Robust GP](13_robust_gp.ipynb)
- [Preference BO](24_pairwise_gp.ipynb)
- [個別Factoryによる候補提案](62_unified_factories.ipynb)